<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/batch_working_capstone_proj_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sec_edgar_downloader



In [ ]:
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = \
    "./triple-mountain-483601-k3-3a823d61bdb7.json"

In [ ]:
!pip install google-genai
from pydantic import BaseModel
from typing import List
from google import genai




class QAPair(BaseModel):
    question: str
    answer: str
    reference: str
    difficulty: str


class QAPairs(BaseModel):
    qa_pairs: List[QAPair]

In [ ]:
import random

def random_flag_percent(random_modulo):
    random_int = random.randint(1, 100)
    #for e.g random_modulo is 5 For 20%
    ret = random_int%(int(random_modulo))
    if(ret == 0):
      return True
    return False

def random_int_range(low, high):
    random_int = random.randint(low, high)
    return random_int

In [ ]:


def create_data_flag(chunk_num, section, total_chunks, num_data_created):
    # Need to take 20% of the chunks at random
    # Keep 100% of top 20% and bottom 20%
    # if there are hundred chunks, we need 20 chunks. 20% of 20 is 4
    # 4 records from top and 4 records from bottom are a must. Remaining
    # 12 records out of 92 at random but if total - num_created is less
    #than the desired number of records, then
    #chunk num 87, num created 8, total 92, desired 12

    #if((total - chunk num) <= (desired - num_created))
         #create the record

    desired_chunks = int(total_chunks*(0.2))
    """if(total_chunks >= 400 and total_chunks < 800):
      desired_chunks = int(total_chunks*(0.1))
    if (total_chunks > 800):
      desired_chunks = int(total_chunks*(0.05))"""
    desired_top_bottom_chunks = int(desired_chunks*(0.2))
    if((chunk_num <= desired_top_bottom_chunks) or
      (chunk_num >= (total_chunks - desired_top_bottom_chunks))):
        return True

    ret =  random_flag_percent(5)
    if(ret == False):
      if((total_chunks - chunk_num) <= (desired_chunks - num_data_created)):
        return True
    return ret





In [ ]:

from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time
from google.api_core.exceptions import ResourceExhausted

def call_model(prompt):
    for retry in range(6):
        try:
            return client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
        except ResourceExhausted:
            wait = min(2 ** retry, 60)
            print(f"Rate limited. Waiting {wait} seconds...")
            time.sleep(wait)

    raise RuntimeError("Too many retries")

In [58]:
#Create json file of all the chunks

!pip install sec-api
!pip install langchain-text-splitters


import requests
import json
import csv
from sec_api import ExtractorApi
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime
from google.colab import files

total_number_of_records = 0
#TEST SECTION
parsed_cutoff_date = datetime.strptime("20201231", "%Y%m%d").date()
#TEST SECTION
sections = {"1", "1A", "7", "8"}
#sections = {"1"}
headers = {
    "User-Agent": "triple-mountain-483601-k3@appspot.gserviceaccount.com"
}

extractor = ExtractorApi(userdata.get('sec_api_key'))

#TEST SECTION
#with open('nasdaq50_cik.csv', mode='r', encoding='utf-8') as file:
with open('nasdaq_cik.csv', mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)

    for row in reader:

        all_ticker_records = []
        ticker_rec_num = 0
        print(row)
        ticker = row[0]
        company  = row[1]
        cik = row[2]
        url = "https://data.sec.gov/submissions/CIK" + cik +".json"

        data = requests.get(url, headers=headers).content
        print(data)
        json_object = json.loads(data)

        recent = json_object["filings"]["recent"]

        forms = recent["form"]
        accessions = recent["accessionNumber"]
        dates = recent["filingDate"]
        #Get info about recent filing dates and types of filings
        for form, accession, date in zip(
          forms,
          accessions,
          dates):
          #Only get filings from last 5 years
          date = date.replace("-","")
          parsed_date = datetime.strptime(date, "%Y%m%d").date()
          if ((form == "10-K") & (parsed_date > parsed_cutoff_date)):
            accession = accession.replace("-","")
            print(date, accession)
            filing_url = "https://www.sec.gov/Archives/edgar/data/" + cik[2:] + "/" + accession + "/" + ticker + "-" + date +".txt"
            for section in sections:
              chunk_file_name = (f"{ticker}-chunks.jsonl")
              file_2 = open(chunk_file_name, "a", encoding="utf-8")
              print(filing_url, section)
              text = extractor.get_section(
                   filing_url,
                    section,
                    "text"
                    )
              metadata = "Reference Ticker-" + ticker + " CompanyName-" + company + " Date-" + date + " Section-" + section
              text_splitter = RecursiveCharacterTextSplitter(
                              separators=[
                              "\n\n",
                              "\n",
                              ". ",
                              " ",
                              ""
                              ],
                              chunk_size=500,
                              chunk_overlap=100
                              )
              chunks = text_splitter.split_text(text)
              total_chunks = len(chunks)
              print("Number of chunks in " + ticker + " " + section + " " + str(total_chunks))
              num_data_created = 0

              record_name = (f"{ticker}-{section}-{date[:4]}")
              #file = open(f"{ticker}-{section}-{date[:4]}.json", "a", encoding="utf-8")
              chunk_num = 0
              for chunk in chunks:

                chunk_record = {}
                chunk_num = chunk_num + 1
                #print(f"processing chunk num {chunk_num}")
                if (create_data_flag(chunk_num, section, total_chunks , num_data_created)):
                  #print(f"chunk num {chunk_num} is picked")
                  #print(f"{chunk}")
                  #put that chunk to be processed

                  chunk_record["chunk_id"] = f"{ticker}-{section}-{date[:4]}-{chunk_num}"
                  #print(f"{chunk_record["chunk_id"]}")
                  chunk_record["ticker"] = ticker
                  chunk_record["company"] = company
                  chunk_record["year"] = date[:4]
                  chunk_record["section"] = section
                  chunk_record["text"] = chunk
                  json.dump(chunk_record,file_2)
                  num_data_created = num_data_created + 1
                  total_number_of_records = total_number_of_records + 1
                  file_2.write("\n")
                  #if(num_data_created >= 10):
                   # break
                  """all_ticker_records.append(chunk_record)
                  total_number_of_records = total_number_of_records + 1
                  ticker_rec_num = ticker_rec_num + 1
        print(f"Records for {ticker} - {ticker_rec_num}")
        chunk_file_name = (f"{ticker}-chunks.json")
        with open(chunk_file_name, "a", encoding="utf-8") as file_2:
          json.dump(all_ticker_records, file_2, indent=4)"""
              file_2.close()

        #files.download(f"{ticker}-chunks.json")
print(f"Total Records - {total_number_of_records}")



['REGN', 'Regeneron Pharmaceuticals', ' Inc.', '0000872589']
b'<?xml version="1.0" encoding="UTF-8"?>\n<Error><Code>NoSuchKey</Code><Message>The specified key does not exist.</Message><Key>submissions/CIK Inc..json</Key><RequestId>ZY475XWB3A9BJNYM</RequestId><HostId>0ssazOg5KBvim6oQAe/cc84+uekw+TeIQTRNgB+3XPgfYLo9bkDtM5ZTFCYsJz3uVvSnoCnYlSg=</HostId></Error>'


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
#Copy data to drive
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '/content/drive/MyDrive'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/capstone_data'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir("source_dir"):
    if filename.endswith('-chunks.jsonl'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

In [55]:
import json

records = []

with open("AMZN-chunks.jsonl", "r") as f:
    for line in f:
        records.append(json.loads(line))

print(len(records))



1161


In [ ]:
#Read the chunks
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '/content/drive/MyDrive'

# 3. Find and read all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('-chunks.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

In [ ]:
!pip install sec-api
!pip install langchain-text-splitters


import requests
import json
import csv
from sec_api import ExtractorApi
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime
from google.colab import files

total_number_of_records = 0
client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
)

#TEST SECTION
parsed_cutoff_date = datetime.strptime("20201231", "%Y%m%d").date()
#parsed_cutoff_date = datetime.strptime("20241231", "%Y%m%d").date()

#TEST SECTION
sections = {"1", "1A", "7", "8"}
#sections = {"1"}
headers = {
    "User-Agent": "triple-mountain-483601-k3@appspot.gserviceaccount.com"
}

extractor = ExtractorApi(userdata.get('sec_api_key'))
positive_records = {}
#TEST SECTION
#with open('nasdaq50_cik.csv', mode='r', encoding='utf-8') as file:
with open('nasdaq10_cik.csv', mode='r', encoding='utf-8') as file:
    reader = csv.reader(file)

    for row in reader:
        print(row)
        ticker = row[0]
        company  = row[1]
        cik = row[2]
        url = "https://data.sec.gov/submissions/CIK" + cik +".json"

        data = requests.get(url, headers=headers).content
        print(data)
        json_object = json.loads(data)

        recent = json_object["filings"]["recent"]

        forms = recent["form"]
        accessions = recent["accessionNumber"]
        dates = recent["filingDate"]
        #Get info about recent filing dates and types of filings
        for form, accession, date in zip(
          forms,
          accessions,
          dates):
          #Only get filings from last 5 years
          date = date.replace("-","")
          parsed_date = datetime.strptime(date, "%Y%m%d").date()
          if ((form == "10-K") & (parsed_date > parsed_cutoff_date)):

            accession = accession.replace("-","")
            print(date, accession)
            filing_url = "https://www.sec.gov/Archives/edgar/data/" + cik[2:] + "/" + accession + "/" + ticker + "-" + date +".txt"
            for section in sections:
              print(filing_url, section)
              text = extractor.get_section(
                   filing_url,
                    section,
                    "text"
                    )
              metadata = "Reference Ticker-" + ticker + " CompanyName-" + company + " Date-" + date + " Section-" + section
              #TODO:REMOVE TEXT
              #file_text = open(f"section.txt", "a", encoding="utf-8")
              #file_text.write(f"{metadata}\n")
              #file_text.write(f"{text}")
              #chunk the section text
              text_splitter = RecursiveCharacterTextSplitter(
                              separators=[
                              "\n\n",
                              "\n",
                              ". ",
                              " ",
                              ""
                              ],
                              chunk_size=500,
                              chunk_overlap=100
                              )

              chunks = text_splitter.split_text(text)
              print("Number of chunks in " + ticker + " " + section + " " +str(len(chunks)))
              chunk_num = 1
              num_data_created = 0
              total_chunks = len(chunks)
              record_name = (f"{ticker}-{section}-{date[:4]}")
              if record_name not in positive_records:
                positive_records[record_name] = []
              file = open(f"{ticker}-{section}-{date[:4]}.json", "a", encoding="utf-8")
              for chunk in chunks:
                #if (section == "8" or create_data_flag(chunk_num, section, total_chunks , num_data_created)):
                if (create_data_flag(chunk_num, section, total_chunks , num_data_created)):

                  prompt = f"""You are a financial analyst creating a training dataset. Given the SEC filing paragraph below with reference:{chunk}
                            Generate:
                             1. Two to three questions answerable solely from this paragraph.
                             2. The exact answer span from the paragraph.
                             3. Difficulty: easy, medium, or hard.
                             use company name {company} and year {date[:4]} when generating question. Return JSON only.
                         """
                  response = client.models.generate_content(
                           model="gemini-2.5-flash",
                           contents=prompt,
                           config={
                              "response_mime_type": "application/json",
                              "response_schema": QAPairs.model_json_schema()
                          }
                        )
                  resp_json = response.model_dump_json()
                  resp_dict = json.loads(resp_json)
                  qa_pairs = resp_dict['parsed']['qa_pairs']
                  for qa_pair in qa_pairs:
                    qa_pair['label'] = "1"
                    qa_pair['metadata'] = metadata
                    #print(qa_pair)
                    #file.write(f"{qa_pair}" + "\n")
                    positive_records[record_name].append(qa_pair)
                    num_data_created = num_data_created + 1
                    total_number_of_records = total_number_of_records + 1
                #TEST SECTION
                #if num_data_created >= 10:
                #  break
                chunk_num = chunk_num+1

              #file.close()
            print(f"Number of records for section: {num_data_created}")
            print(f"Number of records so far: {total_number_of_records}")
            #dump records created so far in a file so as not to loose them
            json.dump(positive_records, file, indent=4)
            file.close()

        data_file_name = (f"{ticker}.json")
        with open(data_file_name, "a", encoding="utf-8") as file_2:
          json.dump(positive_records, file_2, indent=4)
        file_2.close()

        print(f"Total Number Of Records: {total_number_of_records}")
        files.download(f"{ticker}.json")


In [ ]:
#Copy data to drive
import os
import shutil
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

In [ ]:
#Read the data from files

#companies = ["AAPL","MSFT","AMZN","NVDA","META","GOOGL","TSLA","AVGO","COST",
 #            "NFLX","AMD","AMAT","ASML","CSCO","QCOM","INTC","INTU","CMCSA",
  #           "TMUS","TXN","ADBE","PANW","AMGN","SBUX","ISRG","MDLZ","GILD",
   #          "BKNG","REGN","VRTX","ADP","MELI","ADI","KLAC","CTAS","SNPS",
    #         "CDNS","MAR","ORLY","NXPI","CRWD","WDAY","CTSH","ROST","LRCX",
     #        "FAST","PAYX","MCHP","AEP"]
import json

negative_records = {}
negative_records["Negative_Records"] = []
# Open the file in read mode ('r')
with open("output5.json", "r") as file:
    data = json.load(file)

#companies = ["AAPL","MSFT","AMZN","NVDA","META"]
companies = ["AAPL","MSFT"]
#TEST SECTION
years = ["2025", "2024", "2023", "2022", "2021"]
#years = ["2025"]
sections = ["1", "1A", "7", "8"]
#sections = ["1"]
file_records = {}
#Create 25% each of negative records
num_of_each_neg_rec_type = total_number_of_records*(0.95)
i = 0
while i < num_of_each_neg_rec_type:
  #pick 2 companies at ramdom
  company_index_1 = random_int_range(0, (len(companies) - 1 ))
  company_index_2 = random_int_range(0, (len(companies) - 1 ))

  #pick 2 sections at random
  section_index_1 = random_int_range(0, (len(sections) - 1 ))
  section_index_2 = random_int_range(0, (len(sections) - 1 ))
  #pick 2 years at random
  year_index_1 = random_int_range(0, (len(years) - 1 ))
  year_index_2 = random_int_range(0, (len(years) - 1 ))

  #25%
  if(i <  num_of_each_neg_rec_type):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_1] + "-" + years[year_index_1]

  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*2)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*3)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_2]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*4)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_1]
  elif( i >= num_of_each_neg_rec_type and i < (num_of_each_neg_rec_type*5)):
    rec_name_1 = companies[company_index_1]+ "-" + sections[section_index_1] + "-" + years[year_index_1]
    rec_name_2 = companies[company_index_2]+ "-" + sections[section_index_2] + "-" + years[year_index_2]

  records_1 = data[rec_name_1]
  records_2 = data[rec_name_2]

  #pick a random record index from first and second file
  record_index_1 = random_int_range(0, (len(records_1) - 1 ))
  record_index_2 = random_int_range(0, (len(records_2) - 1))



  record_3 = records_1[record_index_1]
  print(records_1[record_index_1])
  record_3['answer'] = records_2[record_index_2]['answer']
  record_3['metadata'] = records_2[record_index_2]['metadata']
  record_3['label'] = "0"
  print(record_3)
  negative_records["Negative_Records"].append(record_3)
  i = i+1







#TODO: write negative record to a file
with open("negative_output2.json", "w") as file_neg:
    json.dump(negative_records, file_neg, indent=4)

print(f"Total Negative Records: {i}")

#pick an index at random based on the size of the data in both
#copy the records and exchange the answers and label 0
#write to file


#create_negative_data_same_section_diff_comapny_diff_year(companies, sections, years)